In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from gru.model import RateGRU
from gru.trainer import Trainer

import importlib
%load_ext autoreload
%autoreload 2

In [3]:
dmat_early = np.load("../vars/dmat_early.npz", allow_pickle=True)
dmat_late  = np.load("../vars/dmat-late.npz", allow_pickle=True)

In [4]:
X_early, Y_early = dmat_early['X'], dmat_early['Y']
X_late,  Y_late  = dmat_late['X'],  dmat_late['Y']

In [5]:
def reshape_XY(X, Y, b_per_trial):
    N = X.shape[0]

    n_full_trials = int(N/b_per_trial)
    
    X = X[:n_full_trials*b_per_trial,:].reshape(n_full_trials, b_per_trial, X.shape[1])
    Y = Y[:n_full_trials*b_per_trial,:].reshape(n_full_trials, b_per_trial, Y.shape[1])
    
    return X, Y

In [6]:
p_train, p_val, p_test = (0.75, 0.15, 0.1)

X_trainval, X_test, Y_trainval, Y_test = train_test_split(X_early, Y_early, test_size=0.2, random_state=1)
X_train,    X_val,  Y_train,    Y_val  = train_test_split(X_trainval, Y_trainval, test_size=p_val/(p_val+p_train), random_state=1)

BWMS        = 10 # 10ms bins
b_per_sec   = int(1000/BWMS)
s_per_trial = 3
b_per_trial = b_per_sec*s_per_trial

X_train, Y_train = reshape_XY(X_train, Y_train, b_per_trial)
X_val,   Y_val   = reshape_XY(X_val, Y_val, b_per_trial)
X_test,  Y_test  = reshape_XY(X_test, Y_test, b_per_trial)


data_early = {'X_train': torch.tensor(X_train, dtype=torch.float32),
              'X_val'  : torch.tensor(X_val, dtype=torch.float32),
              'X_test' : torch.tensor(X_test, dtype=torch.float32),
              'Y_train': torch.tensor(Y_train, dtype=torch.float32),
              'Y_val'  : torch.tensor(Y_val, dtype=torch.float32),
              'Y_test' : torch.tensor(Y_test, dtype=torch.float32)}

In [7]:
gru_early = RateGRU(input_size=X_early.shape[1], hidden_size=100, output_size=Y_early.shape[1], num_layers=1)

In [8]:
t_per_batch = 2

config = {
    'lr': 1e-3,
    'optimizer': optim.AdamW,
    'criterion': nn.MSELoss
}

hparams = {
    'lr_decay': 0.8,
    'batch_size': t_per_batch
}

n_epochs = 150
checkpoint_name = 'fpass'
print_every = 1 # measured in epochs

trainer_early = Trainer(gru_early, data_early, 
                        optim_config=config, 
                        lr_decay=hparams['lr_decay'],
                        batch_size=hparams['batch_size'],
                        n_epochs=n_epochs, 
                        checkpoint_name=checkpoint_name,
                        print_every=print_every,
                        verbose=True
                        )

In [16]:
trainer_early.train()

saving checkpoint to 'fpass_epoch_0.pkl'
saving checkpoint to 'fpass_epoch_0.pkl'
(Epoch 0 / 150) train r2: 0.262; val r2: 0.257; loss: 0.066
saving checkpoint to 'fpass_epoch_1.pkl'
saving checkpoint to 'fpass_epoch_1.pkl'
(Epoch 1 / 150) train r2: 0.270; val r2: 0.264; loss: 0.065
saving checkpoint to 'fpass_epoch_2.pkl'
saving checkpoint to 'fpass_epoch_2.pkl'
(Epoch 2 / 150) train r2: 0.277; val r2: 0.271; loss: 0.065
saving checkpoint to 'fpass_epoch_3.pkl'
saving checkpoint to 'fpass_epoch_3.pkl'
(Epoch 3 / 150) train r2: 0.283; val r2: 0.277; loss: 0.065
saving checkpoint to 'fpass_epoch_4.pkl'
saving checkpoint to 'fpass_epoch_4.pkl'
(Epoch 4 / 150) train r2: 0.288; val r2: 0.282; loss: 0.064
saving checkpoint to 'fpass_epoch_5.pkl'
saving checkpoint to 'fpass_epoch_5.pkl'
(Epoch 5 / 150) train r2: 0.291; val r2: 0.285; loss: 0.064
saving checkpoint to 'fpass_epoch_6.pkl'
saving checkpoint to 'fpass_epoch_6.pkl'
(Epoch 6 / 150) train r2: 0.293; val r2: 0.287; loss: 0.064
saving